# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

**The rule (CTR-fix).** Among pages that are already visible and ranking on page one, some earn far fewer clicks than their position usually gets. Those are title and meta rewrites, not ranking problems. So I score a page by the clicks it leaves on the table: impressions times the gap between its position tier's median CTR and its own CTR, and only when that gap is positive.

- **Reason code:** `low_ctr_for_position`
- **Action:** `rewrite_title_meta`

**Two signals this leans on, checked before I trust the rule.** Both sit behind real FlyRank flags from the session.

1. CTR versus position, behind the CTR-fix flag. If CTR does not track position, the whole "expected CTR per tier" idea falls apart.
2. Staleness versus decline, behind the refresh flag. I check it honestly even though the rule does not use it.

Verdicts are printed from the bucket tables below.

In [1]:
import pandas as pd, numpy as np

df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")
df["is_declining"] = df["trend_direction"].str.lower().eq("down").astype(int)
visible = df[df["impressions_90d"] >= 100]

sig1 = visible.groupby("position_tier").agg(n=("ctr", "size"), mean_ctr=("ctr", "mean")).round(3)
sig1 = sig1.reindex(["top_3", "page_1", "striking", "page_3_5", "deep"])
print("Signal 1 - CTR by position tier (behind the CTR-fix flag):")
print(sig1.to_string())
print("\nVerdict: CONFIRMED. CTR falls steeply below page one (page_1 ~0.36 down to deep ~0.06),")
print("so a tier's median CTR is a fair yardstick for what a page there should earn.")

Signal 1 - CTR by position tier (behind the CTR-fix flag):
                  n  mean_ctr
position_tier                
top_3           533     0.334
page_1         8633     0.355
striking       5903     0.256
page_3_5       6058     0.142
deep            879     0.055

Verdict: CONFIRMED. CTR falls steeply below page one (page_1 ~0.36 down to deep ~0.06),
so a tier's median CTR is a fair yardstick for what a page there should earn.


In [2]:
sig2 = df.groupby("freshness_tier").agg(n=("is_declining", "size"), decline_rate=("is_declining", "mean")).round(3)
sig2 = sig2.reindex(["0-30", "31-90", "91-180", "181+"])
print("Signal 2 - decline rate by staleness (behind the refresh flag):")
print(sig2.to_string())
print("\nVerdict: MIXED. Decline climbs from fresh to about six months, then the 181+ bucket")
print("reverses (small n=174). Staleness alone is not dependable here, so the rule does not use it.")

Signal 2 - decline rate by staleness (behind the refresh flag):
                    n  decline_rate
freshness_tier                     
0-30            20480         0.511
31-90             175         0.589
91-180           9171         0.611
181+              174         0.471

Verdict: MIXED. Decline climbs from fresh to about six months, then the 181+ bucket
reverses (small n=174). Staleness alone is not dependable here, so the rule does not use it.


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

Candidates are visible pages with a real position (`avg_position > 0`) sitting in `top_3`, `page_1`, or `striking`, where a low CTR is fixable by copy rather than by moving the page. The tier's median CTR is the expected bar; score is `impressions_90d * (expected_ctr - ctr)` where that gap is positive. The ranked queue goes to `work/outputs/baseline_action_score.csv`, which stays out of git and regenerates on every run. A small metrics receipt goes to `work/outputs/baseline_metrics.json`, which is committed.

In [3]:
import os, json

good_pos = visible[(visible["avg_position"] > 0) &
                   (visible["position_tier"].isin(["top_3", "page_1", "striking"]))].copy()
expected_ctr = good_pos.groupby("position_tier")["ctr"].median()
good_pos["expected_ctr"] = good_pos["position_tier"].map(expected_ctr)
good_pos["gap"] = good_pos["expected_ctr"] - good_pos["ctr"]

queue = good_pos[good_pos["gap"] > 0].copy()
queue["score"] = queue["impressions_90d"] * queue["gap"]
queue["reason_code"] = "low_ctr_for_position"
queue["action"] = "rewrite_title_meta"
queue = queue.sort_values("score", ascending=False).reset_index(drop=True)

cols = ["content_id", "score", "reason_code", "action", "impressions_90d",
        "avg_position", "position_tier", "ctr", "expected_ctr", "gap"]
os.makedirs("../outputs", exist_ok=True)
queue[cols].to_csv("../outputs/baseline_action_score.csv", index=False)

metrics = {
    "rule": "ctr_fix",
    "reason_code": "low_ctr_for_position",
    "action": "rewrite_title_meta",
    "candidates_scored": int(len(queue)),
    "good_position_visible": int(len(good_pos)),
    "expected_ctr_by_tier": {k: round(float(v), 3) for k, v in expected_ctr.items()},
    "signal_checks": {"ctr_vs_position": "CONFIRMED", "staleness_vs_decline": "MIXED"},
    "score_max": round(float(queue["score"].max()), 1),
}
with open("../outputs/baseline_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)

print("queue rows:", len(queue), " written to work/outputs/baseline_action_score.csv")
print("metrics written to work/outputs/baseline_metrics.json")
queue[cols].head()

queue rows: 7393  written to work/outputs/baseline_action_score.csv
metrics written to work/outputs/baseline_metrics.json


,content_id,score,reason_code,action,impressions_90d,avg_position,position_tier,ctr,expected_ctr,gap
0,content_36ff89c8214e,53117.46,low_ctr_for_position,rewrite_title_meta,295097,7.3,page_1,0.05,0.23,0.18
1,content_c8e9d6ab9013,47995.94,low_ctr_for_position,rewrite_title_meta,208678,9.7,page_1,0.00,0.23,0.23
2,content_5fe46e04994d,46594.35,low_ctr_for_position,rewrite_title_meta,517715,4.2,page_1,0.14,0.23,0.09
3,content_c84a0ab98e90,44654.20,low_ctr_for_position,rewrite_title_meta,223271,7.8,page_1,0.03,0.23,0.20
4,content_8451fc6f034d,43543.04,low_ctr_for_position,rewrite_title_meta,272144,2.3,top_3,0.03,0.19,0.16


## 3. Top-10 review

*For each of the top 10: the action, why it is there, and what would make it wrong.*

Ten highest-scoring pages, each read with a skeptic's eye. The "what would make it wrong" note is the case for skipping the rewrite, so a reviewer knows where the rule can misfire.

In [4]:
def why_wrong(r):
    if r["ctr"] == 0:
        return "CTR reads 0.00, which can be a tracking gap rather than a true miss; verify before editing."
    if r["impressions_90d"] > 100000:
        return "very high impressions often mean a broad or brand query where low CTR is normal, not a title problem."
    if r["gap"] < 0.05:
        return "the gap is small; the current CTR may already suit the query intent."
    return "one strong query can hide a weak average, so the tier median may overstate what this page can reach."

top10 = queue.head(10).copy()
review = pd.DataFrame({
    "rank": range(1, 11),
    "content_id": top10["content_id"].values,
    "action": top10["action"].values,
    "why": [f"{int(i):,} impressions at position {p:.1f}, CTR {c:.2f}% vs tier median {e:.2f}% (gap {g:.2f})"
            for i, p, c, e, g in zip(top10["impressions_90d"], top10["avg_position"],
                                     top10["ctr"], top10["expected_ctr"], top10["gap"])],
    "what_would_make_it_wrong": [why_wrong(r) for _, r in top10.iterrows()],
})
for _, r in review.iterrows():
    print(f"#{r['rank']} {r['content_id']} -> {r['action']}")
    print(f"    why: {r['why']}")
    print(f"    wrong if: {r['what_would_make_it_wrong']}")

#1 content_36ff89c8214e -> rewrite_title_meta
    why: 295,097 impressions at position 7.3, CTR 0.05% vs tier median 0.23% (gap 0.18)
    wrong if: very high impressions often mean a broad or brand query where low CTR is normal, not a title problem.
#2 content_c8e9d6ab9013 -> rewrite_title_meta
    why: 208,678 impressions at position 9.7, CTR 0.00% vs tier median 0.23% (gap 0.23)
    wrong if: CTR reads 0.00, which can be a tracking gap rather than a true miss; verify before editing.
#3 content_5fe46e04994d -> rewrite_title_meta
    why: 517,715 impressions at position 4.2, CTR 0.14% vs tier median 0.23% (gap 0.09)
    wrong if: very high impressions often mean a broad or brand query where low CTR is normal, not a title problem.
#4 content_c84a0ab98e90 -> rewrite_title_meta
    why: 223,271 impressions at position 7.8, CTR 0.03% vs tier median 0.23% (gap 0.20)
    wrong if: very high impressions often mean a broad or brand query where low CTR is normal, not a title problem.
#5 content

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

The weakest picks near the top are the pages where CTR reads exactly 0.00: the score treats them as the biggest misses, but a true zero can be a measurement gap, so they need a human check before any rewrite. The `why_wrong` note flags them.

On leakage: the rule reads only present-state signals (`impressions_90d`, `ctr`, `avg_position`, `position_tier`). It never touches `trend_pct`, `trend_direction`, `is_declining`, or the 30-day comparison windows. `is_declining` appears once above, only to validate the staleness signal, never as a rule input. The check below asserts that.

In [5]:
rule_inputs = ["impressions_90d", "ctr", "avg_position", "position_tier"]
banned = ["trend_pct", "trend_direction", "is_declining",
          "impressions_last_30d", "impressions_prev_30d"]

leaked = [c for c in rule_inputs if c in banned]
print("rule inputs:", rule_inputs)
print("banned (future-window / label-derived):", banned)
print("leaked into the rule:", leaked)
assert not leaked, "a banned column leaked into the rule inputs"

zero_ctr_top = queue.head(20)
zero_ctr_top = zero_ctr_top[zero_ctr_top["ctr"] == 0]
print("\nweak picks in the top 20 (CTR == 0, need a human check):", len(zero_ctr_top))

rule inputs: ['impressions_90d', 'ctr', 'avg_position', 'position_tier']
banned (future-window / label-derived): ['trend_pct', 'trend_direction', 'is_declining', 'impressions_last_30d', 'impressions_prev_30d']
leaked into the rule: []

weak picks in the top 20 (CTR == 0, need a human check): 1


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled: markdown thinking and the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime, Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/`, then submit your repo URL on the card. Done.